In [1]:
# asyncio: 파이썬에서 비동기(Async) 코드를 작성할 수 있게 해주는 표준 라이브러리입니다.
# 이 예제에서는 그래프를 비동기 스트리밍으로 실행하기 위해 사용합니다.
import asyncio

# LangGraph에서 상태 그래프를 만들기 위한 클래스와
# 그래프의 시작/끝을 나타내는 상수를 가져옵니다.
from langgraph.graph import StateGraph, START, END

# Google Cloud Vertex AI 의 Gemini 모델을 사용하기 위한 LangChain 래퍼입니다.
from langchain_google_vertexai import ChatVertexAI

# 대화 메시지(history)를 상태로 다룰 수 있게 해주는 기본 상태 타입입니다.
from langgraph.graph.message import MessagesState

# ToolNode, tools_condition:
# - ToolNode: LLM 이 "도구를 써야 한다"고 판단했을 때 실제로 도구를 실행해 주는 노드
# - tools_condition: LLM 응답 안에 도구 호출이 있는지 보고,
#                    다음에 ToolNode 로 갈지/그냥 넘어갈지 결정하는 조건 함수
from langgraph.prebuilt import ToolNode, tools_condition

# @tool 데코레이터로 파이썬 함수를 LLM 이 쓸 수 있는 "도구"로 등록할 때 사용합니다.
from langchain_core.tools import tool

# AsyncSqliteSaver: LangGraph의 상태(체크포인트)를
# SQLite 데이터베이스에 비동기 방식으로 저장/복원해 주는 도구입니다.
#   → "그래프의 메모리"를 파일(memory.db)에 남기는 역할을 합니다.
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

# -------------------------------
# LLM 정의 (Vertex AI Gemini 모델)
# -------------------------------
# ChatVertexAI 를 통해 Google Vertex AI 의 Gemini 모델을 사용합니다.
# - model_name: 사용할 모델 이름
# - project: GCP 프로젝트 ID
# - location: 리전 (us-central1 등)
# - max_tokens: 한 번에 생성할 최대 토큰 수
llm = ChatVertexAI(
    model_name="gemini-2.5-flash-lite",
    project="ai-prompt-evaluator-489612",
    location="us-central1",
    max_tokens=500
)


c:\my_study\introducing-langgraph\.venv\Lib\site-packages\google\cloud\aiplatform\models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


In [2]:
# -------------------------------
# 상태 정의
# -------------------------------
# MessagesState 를 상속해서, 대화형 상태를 위한 State 를 정의합니다.
# - MessagesState 안에는 보통 "messages" 라는 키에 대화 내역이 들어갑니다.
# - 여기서는 예시로 custom_stuff 라는 추가 필드를 둘 수 있게 했습니다.
class State(MessagesState):
    custom_stuff: str

# 위에서 정의한 State 타입을 사용하는 상태 그래프를 만듭니다.
# 앞으로 노드들을 이 graph_builder 에 등록하고, 간선(실행 순서)을 정의합니다.
graph_builder = StateGraph(State)

In [3]:
# -------------------------------
# Tool 정의
# -------------------------------
# @tool 데코레이터를 사용하면, 일반 파이썬 함수를
# LLM 이 호출할 수 있는 "도구(tool)" 로 등록할 수 있습니다.
# 이 함수는 도시 이름을 입력으로 받아, 단순한 날씨 정보를 문자열로 돌려줍니다.
@tool
def get_weather(city: str):
    """Gets the weather in the given city and returns a simple string."""
    print("TOOL CALLED:", city)  # 실제로 도구가 호출되었는지 콘솔에서 확인하려고 출력
    return f"The weather in {city} is sunny."

# LLM 에게 "너는 get_weather 라는 도구를 쓸 수 있어" 라고 알려주는 단계입니다.
# 이렇게 bind_tools 를 해 두면, LLM 이 응답을 생성할 때
# 필요하다고 판단하면 get_weather 도구를 호출하라는 요청을 만들 수 있습니다.
llm_with_tools = llm.bind_tools(tools=[get_weather])

In [4]:
# -------------------------------
# Chatbot 노드 정의
# -------------------------------
# chatbot 노드는 현재까지의 대화 메시지(state["messages"])를
# 도구 사용이 가능한 LLM(llm_with_tools)에 보내고, 응답을 다시 messages 에 추가합니다.
# - llm_with_tools.invoke(...): LLM 에게 대화 내역을 보내서 응답을 생성
# - LLM 이 도구 호출이 필요하다고 판단하면, 응답 안에 도구 호출 정보가 포함될 수 있습니다.
# - 이 함수는 새로 받은 응답 메시지 하나를 리스트에 담아 반환합니다.
#   (LangGraph 가 기존 messages 와 합쳐서 전체 대화 내역을 관리합니다.)
def chatbot(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# ToolNode 는 위에서 정의한 get_weather 도구를 실제로 실행해 주는 노드입니다.
# LLM 응답 안에 "get_weather 를 호출해 달라"는 정보가 있으면,
# ToolNode 가 그 요청을 읽고, get_weather 함수를 실행한 뒤 결과를 state 에 반영합니다.
tool_node = ToolNode(tools=[get_weather])


In [5]:
# -------------------------------
# 그래프 노드 등록
# -------------------------------
# - "chatbot": LLM 에게 질문을 보내고 응답을 받는 노드
# - "tools": LLM 이 요청한 도구(get_weather)를 실제로 실행하는 ToolNode
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)

# 실행 흐름 정의
# 1) START -> chatbot: 먼저 챗봇 노드가 실행되어, LLM 응답을 만듭니다.
# 2) chatbot 이후에는 tools_condition 에 따라 분기합니다.
#    - LLM 응답 안에 "도구 호출" 정보가 있으면 tools 노드로 이동
#    - 없으면 도구를 거치지 않고 종료할 수 있게 설계하는 패턴입니다.
# 3) tools 노드에서 실제로 도구를 실행한 뒤, 다시 chatbot 으로 돌아와
#    도구 결과를 바탕으로 후속 답변을 생성할 수 있습니다.
graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")

In [6]:
# -------------------------------
# Async 체크포인터와 실행
# -------------------------------
# 이 main 함수는 비동기(async) 방식으로 그래프를 실행하고,
# 실행 도중/실행 후에 체크포인트(메모리)를 확인하는 전체 흐름을 담고 있습니다.
async def main():
    # 기존 memory.db 파일이 있다면 삭제 후 실행 권장
    # (테스트를 처음부터 다시 하고 싶을 때 사용)
    # import os; os.remove("memory.db")  # 필요시 주석 해제

    # AsyncSqliteSaver.from_conn_string("memory.db"):
    # - memory.db 파일을 체크포인터(메모리 저장소)로 사용합니다.
    # - async with 블록이 끝나면 연결을 정리해 줍니다.
    async with AsyncSqliteSaver.from_conn_string("memory.db") as checkpointer:
        # 그래프 컴파일: checkpointer 를 연결해 두면,
        # 실행 중 상태가 자동으로 memory.db 에 저장됩니다.
        graph = graph_builder.compile(checkpointer=checkpointer)

        # 메시지 입력 및 스트리밍 실행
        # - graph.astream(...): 그래프를 비동기 스트림 방식으로 실행하면서
        #   중간중간 업데이트(event)를 하나씩 받아볼 수 있습니다.
        # - config.configurable.thread_id: 이 실행을 "thread_id=2" 인 세션으로 구분합니다.
        async for event in graph.astream(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": "what is the weather in berlin, budapest and bratislava."
                    }
                ]
            },
            stream_mode="updates",
            config={"configurable": {"thread_id": "2"}}
        ):
            # event 는 "어떤 노드에서 어떤 상태 업데이트가 있었는지"를 담고 있는 딕셔너리입니다.
            # 여기서는 각 노드 이름(node_name)별로 마지막 메시지만 골라서 출력합니다.
            for node_name, update in event.items():
                if "messages" in update and update["messages"]:
                    print(node_name, "→", update["messages"][-1].content)

        # 실행 히스토리 확인
        print("\n=== State History ===")
        # aget_state_history: thread_id=2 인 세션에 대해,
        # 과거에 어떤 상태들이 저장되었는지(체크포인트)를 비동기로 하나씩 가져옵니다.
        async for state in graph.aget_state_history({"configurable": {"thread_id": "2"}}):
            # state.next: 그 시점에서 "다음에 어떤 노드가 실행될 예정이었는지" 등
            # LangGraph 가 저장한 상태 정보를 담고 있습니다.
            print(state.next)

# 비동기 main 함수를 실제로 실행합니다.
await main()

chatbot → 
TOOL CALLED: berlin
TOOL CALLED: budapest
TOOL CALLED: bratislava
tools → The weather in bratislava is sunny.
chatbot → The weather in Berlin is sunny, the weather in Budapest is sunny, and the weather in Bratislava is sunny.

=== State History ===
()
('chatbot',)
('tools',)
('chatbot',)
('__start__',)
()
('chatbot',)
('tools',)
('chatbot',)
('__start__',)
